<a href="https://colab.research.google.com/github/smin12345/NLP_Practice/blob/main/NLP_week03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3주차 실습: 한국어 텍스트 전처리

**배포용 반완성 노트북**입니다. 위에서 아래 순서로 실행하고, `TODO`와 체크포인트를 직접 완성하세요.

## 오늘의 목표
1. NFC 유니코드 정규화가 비교·검색·중복 제거에 필요한 이유를 설명한다.
2. URL·이메일·전화번호·반복 문자를 정규표현식으로 탐지하고, 삭제와 마스킹을 구분한다.
3. 공백·정규표현식·형태소 기반 토큰화를 비교한다.
4. Kiwi와 Okt의 결과를 과업에 맞게 해석한다.
5. 불용어·품사 필터가 보존하는 정보와 잃는 정보를 판단한다.

> **제출 원칙:** TODO 실행 결과와 체크포인트 답변을 남기세요. 외부 AI·자료를 사용했다면 사용 범위와 검증 과정을 기록하세요.

> **실행 안내:** 이 노트북은 Kiwi만으로 핵심 실습을 수행할 수 있습니다. Okt는 Java 환경 문제로 설치되지 않을 수 있으므로, 사용할 수 있을 때만 비교합니다.

## 0. 준비 및 실습 데이터

이 데이터는 교육용 예시입니다. 실제 리뷰·게시글을 분석할 때에는 데이터 출처·이용 조건·개인정보 포함 여부를 확인해야 합니다.

In [1]:
# 아래 주석을 해제해 실행하세요.
!pip -q install -U kiwipiepy konlpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 MB 10.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 110.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 26.1 MB/s eta 0:00:00


In [2]:

import re
import unicodedata
from collections import Counter

import pandas as pd
from kiwipiepy import Kiwi

kiwi = Kiwi()

# Okt는 Java 설정에 따라 초기화가 실패할 수 있습니다. 실패해도 Kiwi 실습은 계속할 수 있습니다.
try:
    from konlpy.tag import Okt
    okt = Okt()
    okt_ready = True
except Exception as error:
    okt = None
    okt_ready = False
    print('Okt를 사용할 수 없습니다. Kiwi 중심으로 진행합니다.')
    print('사유:', type(error).__name__)

reviews = [
    (1, "정말 재미있어요!!! 배우 연기가 최고네요 😊", 1),
    (2, "스토리는 평범하지만 음악이 좋아요.", 1),
    (3, "기대했는데 너무너무 지루했어요...", 0),
    (4, "영상미는 훌륭합니다. 그런데 전개가 느려요.", 1),
    (5, "별로예요 ㅠㅠ 다시 보고 싶지 않아요.", 0),
    (6, "문의는 movie@example.com 또는 010-1234-5678로 주세요.", 1),
    (7, "새 소식: https://example.com/movie 를 확인하세요!", 1),
    (8, "가족과 함께 보기 좋은 따뜻한 작품입니다.", 1),
]
df = pd.DataFrame(reviews, columns=['review_id', 'text', 'label'])
df['label_name'] = df['label'].map({1: '긍정', 0: '부정'})
display(df)

,review_id,text,label,label_name
0,1,정말 재미있어요!!! 배우 연기가 최고네요 😊,1,긍정
1,2,스토리는 평범하지만 음악이 좋아요.,1,긍정
2,3,기대했는데 너무너무 지루했어요...,0,부정
3,4,영상미는 훌륭합니다. 그런데 전개가 느려요.,1,긍정
4,5,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,0,부정
5,6,문의는 movie@example.com 또는 010-1234-5678로 주세요.,1,긍정
6,7,새 소식: https://example.com/movie 를 확인하세요!,1,긍정
7,8,가족과 함께 보기 좋은 따뜻한 작품입니다.,1,긍정


## 1. 전처리 파이프라인 이해

전처리는 하나의 정답 함수가 아닙니다. **과업·데이터·평가 기준**에 따라 필요한 규칙이 달라집니다.

`원문 → 유니코드 정규화 → 민감정보/노이즈 탐지 → 처리 정책(보존·마스킹·삭제) → 토큰화 → 품사·불용어 필터 → 텍스트 표현`

- URL·연락처는 단순 노이즈가 아니라 개인정보 또는 참조 정보일 수 있습니다.
- 이모지·반복 문자·느낌표는 감성 신호일 수 있으므로 무조건 삭제하지 않습니다.
- 이후 주차의 빈도·TF-IDF는 여기서 만든 토큰을 입력으로 사용합니다.

### 체크포인트 1

위 파이프라인에서 **반드시 보존해야 할 수 있다고 생각하는 정보**를 하나 고르고, 어떤 분석 과업에서 유용한지 1~2문장으로 설명하세요.

반복 문자와 느낌표는 보존할 필요가 있습니다. 감성 분석에서 ㅋㅋㅋㅋ, 정말 좋아요!!! 같은 표현은 사용자의 긍정·부정 감정과 감정의 강도를 파악하는 데 유용하기 때문입니다.

## 2. 유니코드 정규화: 겉보기와 내부 표현

겉보기에는 같은 한글도 내부 유니코드 표현이 달라 비교·중복 제거에 문제가 생길 수 있습니다. 분석 전에는 보통 NFC 형태로 통일합니다.

In [3]:
composed = '가'
decomposed = unicodedata.normalize('NFD', composed)
print('겉보기:', composed, decomposed)
print('정규화 전 동일 여부:', composed == decomposed)

# TODO 1. composed와 decomposed를 각각 NFC로 정규화하고, 동일한지 출력하세요.
# nfc_composed = unicodedata.normalize('NFC', composed)
# nfc_decomposed = unicodedata.normalize('NFC', decomposed)
# print('NFC 정규화 후 동일 여부:', nfc_composed == nfc_decomposed)

겉보기: 가 가
정규화 전 동일 여부: False


In [4]:
# TODO 2. text 열 전체를 NFC로 정규화한 text_nfc 열을 만드세요.
# 힌트: apply()함수와 lambda 식을 이용하세요
# df['text_nfc'] = ...
df['text_nfc'] = df['text'].apply(
    lambda text: unicodedata.normalize('NFC', text)
)

if 'text_nfc' in df.columns:
    display(df[['review_id', 'text', 'text_nfc']])
else:
    print('TODO 2를 완료한 뒤 다시 실행하세요.')

,review_id,text,text_nfc
0,1,정말 재미있어요!!! 배우 연기가 최고네요 😊,정말 재미있어요!!! 배우 연기가 최고네요 😊
1,2,스토리는 평범하지만 음악이 좋아요.,스토리는 평범하지만 음악이 좋아요.
2,3,기대했는데 너무너무 지루했어요...,기대했는데 너무너무 지루했어요...
3,4,영상미는 훌륭합니다. 그런데 전개가 느려요.,영상미는 훌륭합니다. 그런데 전개가 느려요.
4,5,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,별로예요 ㅠㅠ 다시 보고 싶지 않아요.
5,6,문의는 movie@example.com 또는 010-1234-5678로 주세요.,문의는 movie@example.com 또는 010-1234-5678로 주세요.
6,7,새 소식: https://example.com/movie 를 확인하세요!,새 소식: https://example.com/movie 를 확인하세요!
7,8,가족과 함께 보기 좋은 따뜻한 작품입니다.,가족과 함께 보기 좋은 따뜻한 작품입니다.


### 체크포인트 2

NFC 정규화를 하지 않으면 **중복 제거**, **검색**, **모델 입력** 중 하나에서 어떤 문제가 발생할 수 있는지 예를 들어 설명하세요.

NFC 정규화를 하지 않으면 화면상으로 동일한 한글도 내부 유니코드 조합이 달라서 중복 데이터로 인식되지 않을 수 있습니다. 예를 들어 두 개의 가가 각각 완성형과 자모 결합형으로 저장되어 있으면 중복 제거 과정에서 서로 다른 문자열로 처리될 수 있습니다.

## 3. 노이즈·개인정보 패턴 탐지와 처리 정책

정규표현식은 일정한 형식의 문자열을 찾는 도구입니다. 아래 탐지는 교육용 예시이며, 실제 개인정보 탐지에는 더 엄격한 정책·검토가 필요합니다. 실습에서는 연락처를 즉시 삭제하지 않고 **마스킹**해 정보의 유형을 남깁니다.

In [5]:
def inspect_patterns(text):
    flags = []
    if re.search(r'https?://\S+|www\.\S+', text):
        flags.append('URL')
    if re.search(r'[\w.+-]+@[\w-]+\.[\w.-]+', text):
        flags.append('이메일 가능성')
    # TODO 3. 전화번호처럼 보이는 패턴을 탐지하세요.
    # 힌트: r'\d{2,3}[- ]?\d{3,4}[- ]?\d{4}'
    if re.search(r'\d{2,3}[- ]?\d{3,4}[- ]?\d{4}', text):
        flags.append('전화번호 가능성')



    # TODO 4. 같은 문자가 3회 이상 반복되는 패턴을 탐지하세요. 예: ㅋㅋㅋㅋ, !!!!
    # 힌트: r'(.)\1{2,}'
    if re.search(r'(.)\1{2,}', text):
        flags.append('반복 문자')



    return ', '.join(flags) if flags else '특이 패턴 없음'

if 'text_nfc' in df.columns:
    df['pattern_flags'] = df['text_nfc'].apply(inspect_patterns)
    display(df[['review_id', 'text_nfc', 'pattern_flags']])
else:
    print('TODO 2를 완료한 뒤 다시 실행하세요.')

,review_id,text_nfc,pattern_flags
0,1,정말 재미있어요!!! 배우 연기가 최고네요 😊,반복 문자
1,2,스토리는 평범하지만 음악이 좋아요.,특이 패턴 없음
2,3,기대했는데 너무너무 지루했어요...,반복 문자
3,4,영상미는 훌륭합니다. 그런데 전개가 느려요.,특이 패턴 없음
4,5,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,특이 패턴 없음
5,6,문의는 movie@example.com 또는 010-1234-5678로 주세요.,"이메일 가능성, 전화번호 가능성"
6,7,새 소식: https://example.com/movie 를 확인하세요!,URL
7,8,가족과 함께 보기 좋은 따뜻한 작품입니다.,특이 패턴 없음


In [6]:
def basic_clean(text, mask_sensitive=True):
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' [URL] ' if mask_sensitive else ' ', text)
    text = re.sub(r'[\w.+-]+@[\w-]+\.[\w.-]+', ' [EMAIL] ' if mask_sensitive else ' ', text)
    text = re.sub(r'\d{2,3}[- ]?\d{3,4}[- ]?\d{4}', ' [PHONE] ' if mask_sensitive else ' ', text)
    # TODO 5. 연속 공백을 하나로 줄이고 앞뒤 공백을 제거하세요.
    text = re.sub(r'\s+', ' ', text).strip()


    return text

if 'text_nfc' in df.columns:
    df['clean_text'] = df['text_nfc'].apply(basic_clean)
    display(df[['text_nfc', 'clean_text']])
else:
    print('TODO 2를 완료한 뒤 다시 실행하세요.')

,text_nfc,clean_text
0,정말 재미있어요!!! 배우 연기가 최고네요 😊,정말 재미있어요!!! 배우 연기가 최고네요 😊
1,스토리는 평범하지만 음악이 좋아요.,스토리는 평범하지만 음악이 좋아요.
2,기대했는데 너무너무 지루했어요...,기대했는데 너무너무 지루했어요...
3,영상미는 훌륭합니다. 그런데 전개가 느려요.,영상미는 훌륭합니다. 그런데 전개가 느려요.
4,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,별로예요 ㅠㅠ 다시 보고 싶지 않아요.
5,문의는 movie@example.com 또는 010-1234-5678로 주세요.,문의는 [EMAIL] 또는 [PHONE] 로 주세요.
6,새 소식: https://example.com/movie 를 확인하세요!,새 소식: [URL] 를 확인하세요!
7,가족과 함께 보기 좋은 따뜻한 작품입니다.,가족과 함께 보기 좋은 따뜻한 작품입니다.


### 체크포인트 3

반복 문자 또는 이모지를 일괄 삭제할 때 생길 수 있는 **정보 손실** 사례를 한 가지 작성하세요. 전화번호·이메일을 탐지한 뒤 실제 서비스에서 추가로 고려해야 할 조치도 한 가지 적으세요.

ㅋㅋㅋㅋ, ㅠㅠㅠ, 이모지 등을 일괄 삭제하면 사용자의 감정 종류와 강도를 파악할 수 없어 감성 분석 결과가 부정확해질 수 있습니다. 전화번호와 이메일을 탐지한 뒤에는 개인정보 보호를 위해 [PHONE], [EMAIL] 등의 문자로 마스킹하여 원본 정보가 노출되지 않도록 해야 합니다.

## 4. 토큰화 비교: 공백·정규표현식·형태소 분석

공백 기준 토큰화는 간단하지만 조사·어미가 결합된 한국어의 문법 정보를 충분히 분리하지 못합니다. Okt가 준비되지 않은 경우에도 Kiwi 결과와 공백 토큰을 비교해 관찰하세요.

In [7]:
if 'clean_text' not in df.columns:
    print('TODO 5를 완료한 뒤 다시 실행하세요.')
else:
    sample = df.loc[0, 'clean_text']
    print('원문:', sample)
    print('공백 토큰:', sample.split())
    print('정규표현식 토큰:', re.findall(r'[가-힣A-Za-z0-9]+', sample))
    print('\nKiwi 형태소:', [(token.form, token.tag) for token in kiwi.tokenize(sample)])
    if okt_ready:
        print('Okt 형태소:', okt.pos(sample))
    else:
        print('Okt 형태소: 현재 환경에서는 비교 생략')

원문: 정말 재미있어요!!! 배우 연기가 최고네요 😊
공백 토큰: ['정말', '재미있어요!!!', '배우', '연기가', '최고네요', '😊']
정규표현식 토큰: ['정말', '재미있어요', '배우', '연기가', '최고네요']

Kiwi 형태소: [('정말', 'MAG'), ('재미있', 'VA'), ('어요', 'EF'), ('!!!', 'SF'), ('배우', 'NNG'), ('연기', 'NNG'), ('가', 'JKS'), ('최고', 'NNG'), ('이', 'VCP'), ('네요', 'EF'), ('😊', 'W_EMOJI')]
Okt 형태소: [('정말', 'Noun'), ('재미있어요', 'Adjective'), ('!!!', 'Punctuation'), ('배우', 'Noun'), ('연기', 'Noun'), ('가', 'Josa'), ('최고', 'Noun'), ('네', 'Suffix'), ('요', 'Josa'), ('😊', 'Foreign')]


In [10]:
# TODO 6. 아래 문장을 관심 있는 주제로 바꾸고, Kiwi와 (가능하면) Okt 결과를 비교하세요.
my_sentence = '인공지능을 활용하면 축구 경기 데이터를 빠르게 분석할 수 있다.'
print('문장:', my_sentence)
print('Kiwi:', [(t.form, t.tag) for t in kiwi.tokenize(my_sentence)])
if okt_ready:
    print('Okt :', okt.pos(my_sentence))
    pass
else:
    print('Okt : 현재 환경에서는 비교 생략')

문장: 인공지능을 활용하면 축구 경기 데이터를 빠르게 분석할 수 있다.
Kiwi: [('인공', 'NNG'), ('지능', 'NNG'), ('을', 'JKO'), ('활용', 'NNG'), ('하', 'XSV'), ('면', 'EC'), ('축구', 'NNG'), ('경기', 'NNG'), ('데이터', 'NNG'), ('를', 'JKO'), ('빠르', 'VA'), ('게', 'EC'), ('분석', 'NNG'), ('하', 'XSV'), ('ᆯ', 'ETM'), ('수', 'NNB'), ('있', 'VA'), ('다', 'EF'), ('.', 'SF')]
Okt : [('인공', 'Noun'), ('지능', 'Noun'), ('을', 'Josa'), ('활용', 'Noun'), ('하면', 'Verb'), ('축구', 'Noun'), ('경기', 'Noun'), ('데이터', 'Noun'), ('를', 'Josa'), ('빠르게', 'Adjective'), ('분석', 'Noun'), ('할', 'Verb'), ('수', 'Noun'), ('있다', 'Adjective'), ('.', 'Punctuation')]


### 체크포인트 4

본인 문장에서 Kiwi와 Okt가 다르게 나눈 토큰 또는 다르게 태깅한 사례를 하나 기록하세요. Okt를 사용할 수 없었다면 Kiwi 결과에서 조사·어미가 어떻게 분리되었는지 기록하세요. 결과의 우열이 아니라, **어떤 과업에 더 편리할지**를 함께 설명하세요.

Kiwi는 활용하면을 활용(NNG), 하(XSV), 면(EC)으로 세분화했지만, Okt는 활용(Noun), 하면(Verb)으로 나누었다. Kiwi는 어간과 어미를 자세히 분석하는 문법적 분석에 편리하고, Okt는 형태소 구분이 비교적 단순하여 감성 분석이나 단어 빈도 분석에 편리할 수 있다.

## 5. 품사 선택과 불용어 처리

키워드 추출에는 명사 중심 토큰이 유용할 수 있습니다. 반면 감성 분석에서는 부정 표현·부사·어미가 중요할 수 있습니다. 아래에서는 먼저 명사를 추출한 뒤, **도메인 불용어**를 적용합니다. 조사는 이미 품사 필터에서 제거되므로, 명사 목록에 조사 불용어를 넣어도 효과가 거의 없다는 점을 확인하세요.

In [12]:
def kiwi_nouns(text):
    # TODO 7. Kiwi 토큰 중 품사 태그가 N으로 시작하는 토큰의 form만 반환하세요.
    # 힌트: [token.form for token in kiwi.tokenize(text) if token.tag.startswith('N')]

    return [
        token.form
        for token in kiwi.tokenize(text)
        if token.tag.startswith('N')
    ]

if 'clean_text' in df.columns:
    df['nouns'] = df['clean_text'].apply(kiwi_nouns)
    display(df[['review_id', 'clean_text', 'nouns']])
else:
    print('TODO 5를 완료한 뒤 다시 실행하세요.')

,review_id,clean_text,nouns
0,1,정말 재미있어요!!! 배우 연기가 최고네요 😊,"[배우, 연기, 최고]"
1,2,스토리는 평범하지만 음악이 좋아요.,"[스토리, 음악]"
2,3,기대했는데 너무너무 지루했어요...,[기대]
3,4,영상미는 훌륭합니다. 그런데 전개가 느려요.,"[영상미, 전개]"
4,5,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,[]
5,6,문의는 [EMAIL] 또는 [PHONE] 로 주세요.,[문의]
6,7,새 소식: [URL] 를 확인하세요!,"[소식, 확인]"
7,8,가족과 함께 보기 좋은 따뜻한 작품입니다.,"[가족, 작품]"


In [13]:
# 과업에 따라 수정할 수 있는 도메인 불용어 예시입니다.
stopwords = {'영화', '작품'}

# TODO 8. nouns에서 stopwords에 없는 토큰만 남긴 filtered_nouns 열을 만드세요.
# 힌트: [token for token in tokens if token not in stopwords]
# df['filtered_nouns'] = df['nouns'].apply(lambda tokens: ...)
df['filtered_nouns'] = df['nouns'].apply(
    lambda tokens: [token for token in tokens if token not in stopwords]
)

if 'filtered_nouns' in df.columns:
    display(df[['clean_text', 'nouns', 'filtered_nouns']])
else:
    print('TODO 8을 완료한 뒤 다시 실행하세요.')

,clean_text,nouns,filtered_nouns
0,정말 재미있어요!!! 배우 연기가 최고네요 😊,"[배우, 연기, 최고]","[배우, 연기, 최고]"
1,스토리는 평범하지만 음악이 좋아요.,"[스토리, 음악]","[스토리, 음악]"
2,기대했는데 너무너무 지루했어요...,[기대],[기대]
3,영상미는 훌륭합니다. 그런데 전개가 느려요.,"[영상미, 전개]","[영상미, 전개]"
4,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,[],[]
5,문의는 [EMAIL] 또는 [PHONE] 로 주세요.,[문의],[문의]
6,새 소식: [URL] 를 확인하세요!,"[소식, 확인]","[소식, 확인]"
7,가족과 함께 보기 좋은 따뜻한 작품입니다.,"[가족, 작품]",[가족]


In [14]:
# TODO 9. filtered_nouns의 모든 토큰을 모아 빈도를 계산하고 상위 10개를 출력하세요.
# all_nouns = [token for tokens in df['filtered_nouns'] for token in tokens]
# noun_freq = Counter(all_nouns)
# display(pd.DataFrame(noun_freq.most_common(10), columns=['token', 'count']))
all_nouns = [
    token
    for tokens in df['filtered_nouns']
    for token in tokens
]

noun_freq = Counter(all_nouns)

display(
    pd.DataFrame(
        noun_freq.most_common(10),
        columns=['token', 'count']
    )
)

,token,count
0,배우,1
1,연기,1
2,최고,1
3,스토리,1
4,음악,1
5,기대,1
6,영상미,1
7,전개,1
8,문의,1
9,소식,1


### 체크포인트 5

`좋지 않아요`, `너무 지루했어요`와 같은 감성 표현을 생각해 보세요. 명사만 남기면 어떤 정보가 사라질 수 있나요? 감성 분석과 키워드 추출 중 어느 과업에 명사 중심 처리가 더 적합한지 이유와 함께 답하세요.

명사만 남기면 좋지 않아요의 부정 표현이나 너무 지루했어요의 감정과 강도 정보가 사라질 수 있습니다. 명사 중심 처리는 문서의 주요 주제와 핵심 단어를 찾는 키워드 추출에 더 적합하며, 감성 분석에서는 형용사·동사·부정어·부사도 함께 보존해야 합니다.

## 6. 전처리 전·후 비교

전처리의 효과는 단순히 “짧아졌다”가 아니라, 과업에 필요한 정보가 적절히 남았는지로 판단해야 합니다.

In [ ]:
if 'filtered_nouns' in df.columns:
    comparison = df[['review_id', 'text', 'clean_text', 'filtered_nouns']].copy()
    comparison['original_length'] = comparison['text'].str.len()
    comparison['clean_length'] = comparison['clean_text'].str.len()
    display(comparison)
else:
    print('TODO 8을 완료한 뒤 다시 실행하세요.')

# TODO 10. 아래 질문에 Markdown 셀로 답하세요.
# 1) 한 리뷰를 골라 전처리 전·후 달라진 부분을 설명하세요.
# 2) 해당 변화가 목적(키워드 추출 또는 감성 분석)에 적합한지 판단하세요.
# 3) 삭제하지 말아야 했을 수 있는 정보가 있다면 적으세요.

1. URL이나 전화번호가 포함된 리뷰를 선택하였다. 전처리 후 URL은 `[URL]`, 전화번호는 `[PHONE]`으로 마스킹되었으며, 연속된 공백도 하나로 정리되었다. 또한 명사 추출 과정에서 조사와 어미가 제외되고 핵심 명사만 남았다.

2. 이러한 변화는 개인정보와 불필요한 표현을 제외하고 핵심 단어를 확인할 수 있으므로 키워드 추출 목적에 적합하다.

3. 감성 분석이 목적이라면 `좋다`, `지루하다` 같은 감정 표현과 `너무` 같은 감정의 강도를 나타내는 표현, 이모지와 반복 문자는 삭제하지 않아야 할 수 있다.

## 제출 체크리스트

- [ ] TODO 1~10을 작성하고 실행했다.
- [ ] NFC 정규화의 필요성을 설명했다.
- [ ] URL·이메일·전화번호·반복 문자 중 2개 이상을 탐지했다.
- [ ] 민감정보를 마스킹할 때와 삭제할 때의 차이를 설명했다.
- [ ] 공백·정규표현식·Kiwi·(가능하면) Okt 토큰화 결과를 비교했다.
- [ ] 명사 중심 처리와 도메인 불용어 처리의 한계를 설명했다.
- [ ] 전처리 전·후 비교 결과와 판단 근거를 기록했다.
- [ ] 외부 AI/자료 사용 시 사용 범위와 검증 과정을 기록했다.

### 심화 과제(선택)
1. `basic_clean`에서 이모지·반복 문자 수를 별도 특징으로 보존하세요.
2. 감성 표현을 보존하려면 명사 외에 어떤 품사를 남겨야 하는지 규칙을 제안·구현하세요.
3. 직접 수집한 텍스트 5건 이상에 같은 규칙을 적용하고, 출처·라이선스·개인정보 처리 방침을 기록하세요.

In [18]:
def advanced_clean(text):
    text = unicodedata.normalize('NFC', text)

    # 이모지와 반복 문자 개수 저장
    emoji_count = len(re.findall(r'[\U0001F300-\U0001FAFF]', text))
    repeat_count = len(re.findall(r'(.)\1{2,}', text))

    text = re.sub(r'https?://\S+|www\.\S+', ' [URL] ', text)
    text = re.sub(r'[\w.+-]+@[\w-]+\.[\w.-]+', ' [EMAIL] ', text)
    text = re.sub(r'\d{2,3}[- ]?\d{3,4}[- ]?\d{4}', ' [PHONE] ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return pd.Series([text, emoji_count, repeat_count])


df[['advanced_clean_text', 'emoji_count', 'repeat_count']] = (
    df['text'].apply(advanced_clean)
)

display(
    df[['text', 'advanced_clean_text', 'emoji_count', 'repeat_count']]
)
def kiwi_sentiment_tokens(text):
    tokens = kiwi.tokenize(text)

    return [
        token.form
        for token in tokens
        if token.tag.startswith(('N', 'V', 'MA'))
    ]


df['sentiment_tokens'] = df['clean_text'].apply(kiwi_sentiment_tokens)

display(df[['clean_text', 'sentiment_tokens']])
my_reviews = pd.DataFrame({
    'text': [
        '배우들의 연기가 정말 좋았어요!!! 😊',
        '스토리가 너무 지루해서 아쉬웠어요ㅠㅠㅠ',
        '마지막 반전 장면이 정말 충격적이었습니다😲',
        '영상미는 좋았지만 결말은 마음에 들지 않았어요.',
        '오랜만에 재미있게 본 영화라서 추천합니다!!!'
    ]
})

my_reviews[
    ['clean_text', 'emoji_count', 'repeat_count']
] = my_reviews['text'].apply(advanced_clean)

my_reviews['sentiment_tokens'] = (
    my_reviews['clean_text'].apply(kiwi_sentiment_tokens)
)

display(my_reviews)

,text,advanced_clean_text,emoji_count,repeat_count
0,정말 재미있어요!!! 배우 연기가 최고네요 😊,정말 재미있어요!!! 배우 연기가 최고네요 😊,1,1
1,스토리는 평범하지만 음악이 좋아요.,스토리는 평범하지만 음악이 좋아요.,0,0
2,기대했는데 너무너무 지루했어요...,기대했는데 너무너무 지루했어요...,0,1
3,영상미는 훌륭합니다. 그런데 전개가 느려요.,영상미는 훌륭합니다. 그런데 전개가 느려요.,0,0
4,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,0,0
5,문의는 movie@example.com 또는 010-1234-5678로 주세요.,문의는 [EMAIL] 또는 [PHONE] 로 주세요.,0,0
6,새 소식: https://example.com/movie 를 확인하세요!,새 소식: [URL] 를 확인하세요!,0,0
7,가족과 함께 보기 좋은 따뜻한 작품입니다.,가족과 함께 보기 좋은 따뜻한 작품입니다.,0,0


,clean_text,sentiment_tokens
0,정말 재미있어요!!! 배우 연기가 최고네요 😊,"[정말, 재미있, 배우, 연기, 최고, 이]"
1,스토리는 평범하지만 음악이 좋아요.,"[스토리, 음악, 좋]"
2,기대했는데 너무너무 지루했어요...,"[기대, 너무, 너무]"
3,영상미는 훌륭합니다. 그런데 전개가 느려요.,"[영상미, 그런데, 전개, 느리]"
4,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,"[별로, 이, 다시, 보, 싶, 않]"
5,문의는 [EMAIL] 또는 [PHONE] 로 주세요.,"[문의, 또는, 주]"
6,새 소식: [URL] 를 확인하세요!,"[소식, 확인]"
7,가족과 함께 보기 좋은 따뜻한 작품입니다.,"[가족, 함께, 보, 좋, 작품, 이]"


,text,clean_text,emoji_count,repeat_count,sentiment_tokens
0,배우들의 연기가 정말 좋았어요!!! 😊,배우들의 연기가 정말 좋았어요!!! 😊,1,1,"[배우, 연기, 정말, 좋]"
1,스토리가 너무 지루해서 아쉬웠어요ㅠㅠㅠ,스토리가 너무 지루해서 아쉬웠어요ㅠㅠㅠ,0,1,"[스토리, 너무, 아쉽]"
2,마지막 반전 장면이 정말 충격적이었습니다😲,마지막 반전 장면이 정말 충격적이었습니다😲,1,0,"[마지막, 반전, 장면, 정말, 충격, 이]"
3,영상미는 좋았지만 결말은 마음에 들지 않았어요.,영상미는 좋았지만 결말은 마음에 들지 않았어요.,0,0,"[영상미, 좋, 결말, 마음, 들, 않]"
4,오랜만에 재미있게 본 영화라서 추천합니다!!!,오랜만에 재미있게 본 영화라서 추천합니다!!!,0,1,"[오랜만, 재미있, 보, 영화, 이, 추천]"


### 출처·라이선스·개인정보 처리

사용한 텍스트 5건은 과제를 위해 직접 작성한 가상의 영화 리뷰이다. 외부 데이터나 저작물을 사용하지 않았으며, 과제 작성자가 직접 작성한 문장이므로 과제 분석 목적으로 사용할 수 있다.

리뷰에는 실제 이름, 전화번호, 이메일 등의 개인정보를 포함하지 않았다. URL이나 연락처가 포함되는 경우에는 원문을 그대로 노출하지 않고 `[URL]`, `[EMAIL]`, `[PHONE]`으로 마스킹하는 개인정보 처리 방침을 적용하였다.

## 다음 주차 예고

다음 주차에는 전처리된 텍스트를 컴퓨터가 계산할 수 있는 수치 벡터로 표현합니다. 단어 빈도, Bag-of-Words, TF-IDF를 이용해 문서를 비교하고 분류 모델의 입력 특징을 구성합니다.